# Mamba + Workspace POC — Kaggle T4 x2

Run cells B and D (the go/no-go pair) **in parallel** on two T4 GPUs.

**Setup before running:**
1. Settings → Accelerator → **GPU T4 x2** (must select dual GPU)
2. Settings → Internet → **On** (needed for pip install + git clone)
3. Add a Kaggle Secret called `github_token` with a GitHub Personal Access Token
   that has `repo` scope (Settings → Secrets → Add Secret)

All training logic lives in `kaggle_runner.py` in the repo — when you `git pull`,
the code updates automatically. The notebook just clones, installs deps, and calls the script.

In [ ]:
# Install dependencies — pure-PyTorch path, no mamba-ssm needed
!pip install -q einops pyyaml wandb numpy

In [ ]:
import os
import subprocess
import shutil

# Clone the private repo using a GitHub token stored in Kaggle Secrets
# Before running: add a Kaggle Secret named "github_token" with a GitHub PAT (repo scope)
# Settings → Secrets → Add Secret → Name: github_token, Value: <your PAT>

repo_root = '/kaggle/working/jasper'
work_dir = os.path.join(repo_root, 'mamba-poc')
repo_url = 'https://github.com/rbf22/jasper.git'

# Clean up stale state from previous runs (e.g. moved folders)
stale = ['/kaggle/working/mamba-poc']
for s in stale:
    if os.path.exists(s):
        print(f"Removing stale dir: {s}")
        shutil.rmtree(s)

if os.path.exists(os.path.join(repo_root, '.git')):
    # Already cloned — pull latest and restore any missing files
    print(f"Repo exists, pulling latest...")
    subprocess.run(['git', '-C', repo_root, 'reset', '--hard', 'HEAD'], check=True)
    subprocess.run(['git', '-C', repo_root, 'pull'], check=True)
else:
    # Remove partial clone if any
    if os.path.exists(repo_root):
        shutil.rmtree(repo_root)
    # Clone with token auth from Kaggle Secrets
    try:
        import kaggle_secrets
        user_secrets = kaggle_secrets.UserSecretsClient()
        gh_token = user_secrets.get_secret("github_token")
        authed_url = repo_url.replace('https://', f'https://{gh_token}@')
        print(f"Cloning private repo with token auth...")
        subprocess.run(['git', 'clone', authed_url, repo_root], check=True)
    except Exception as e:
        print(f"Kaggle Secrets auth failed: {e}")
        print("Falling back to public clone (will fail if repo is private)...")
        subprocess.run(['git', 'clone', repo_url, repo_root], check=True)

os.chdir(work_dir)
print(f"Working directory: {os.getcwd()}")
print("Files:", os.listdir('.'))

In [ ]:
# Verify GPU and environment
import torch
n_gpus = torch.cuda.device_count()
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPUs visible: {n_gpus}")
for i in range(n_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
print(f"Torch: {torch.__version__}")

assert n_gpus >= 2, f"Need 2 GPUs for parallel training, found {n_gpus}. Set Accelerator → GPU T4 x2."

In [ ]:
# Quick data test — verifies all 3 task generators
!python data.py

In [ ]:
# Param count check — verify all 4 cells are ~30M
!python model.py

In [ ]:
# Smoke test — verifies both cells train correctly (~5 min on T4)
!python kaggle_runner.py --smoke-test

## Train Cell B + Cell D in Parallel (GPU 0 + GPU 1)

`kaggle_runner.py` handles everything: launches both cells on separate GPUs, stays alive with periodic status updates, and auto-saves checkpoints to `/kaggle/working/` when done.

- Use `--clean` to start fresh (deletes old checkpoints)
- Without `--clean`, auto-resumes from last checkpoint if session died

~12 hours total wall time.

In [ ]:
# Train both cells in parallel — stays alive with periodic updates
# Add --clean if you want to start fresh (deletes old checkpoints)
!python kaggle_runner.py --clean

## Monitor (optional)

If training is running in a different cell, you can't run this simultaneously (Kaggle blocks). But if training crashed or you're in a new session, run this to check logs.

In [ ]:
# Check status and recent logs
!python kaggle_runner.py --status

## Analysis (R2, R3, R4)

Run after training completes. Uses Cell D's checkpoint.

In [ ]:
# Run all analyses on Cell D
# R2: K sweep (accuracy vs inference K at each depth)
# R3: Linear probes (decodability of workspace slots)
# R4: Selective ablation (J-space signature)
!python probe.py --checkpoint checkpoints/cellD_latest.pt --config configs/cell_d_kaggle.yaml --all

In [ ]:
## Save Outputs

Copies checkpoints, probe results, and training logs to `/kaggle/working/` so they survive session end. Also runs automatically after training completes.

In [ ]:
# Save all outputs to /kaggle/working/ (also runs automatically after training)
!python kaggle_runner.py --save-outputs